In [1]:
# Cell 1: Install thư viện
!pip install sentence-transformers scikit-learn pandas numpy implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=933264 sha256=271deaefa7bfff3429cc4381a1fadde6f276f204eec31fbf0083da477bb4f4c4
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [2]:
!pip install -q pymysql sqlalchemy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 1.4 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
import pickle
import json

# Load data
df_orders = pd.read_csv('/content/orders_behavior.csv')
df_wishlist = pd.read_csv('/content/wishlist.csv')
df_products = pd.read_csv('/content/products.csv')

print(f"Orders: {len(df_orders)} rows")
print(f"Wishlist: {len(df_wishlist)} rows")
print(f"Products: {len(df_products)} rows")
df_products.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/orders_behavior.csv'

In [ ]:
# Tính điểm tương tác tổng hợp cho mỗi cặp user-product
# Công thức: purchase_score + rating_bonus + wishlist_bonus
def compute_interaction_scores(df_orders, df_wishlist):
    interactions = []

    # Từ orders: mua nhiều lần = điểm cao hơn, đánh giá cao = bonus
    for _, row in df_orders.iterrows():
        purchase_score = min(row['purchase_count'] * 2.0, 10.0)  # Cap ở 10
        rating_bonus = (row['user_rating'] / 5.0) * 3.0 if row['user_rating'] > 0 else 0
        interactions.append({
            'user_id': row['user_id'],
            'product_id': row['product_id'],
            'score': purchase_score + rating_bonus,
            'source': 'purchase'
        })

    # Từ wishlist: yêu thích = 3 điểm
    for _, row in df_wishlist.iterrows():
        interactions.append({
            'user_id': row['user_id'],
            'product_id': row['product_id'],
            'score': 3.0,
            'source': 'wishlist'
        })

    df_inter = pd.DataFrame(interactions)
    # Nếu user vừa mua vừa wishlist cùng product → cộng dồn, cap ở 15
    df_inter = df_inter.groupby(['user_id', 'product_id'])['score'].sum().reset_index()
    df_inter['score'] = df_inter['score'].clip(upper=15.0)
    return df_inter

df_interactions = compute_interaction_scores(df_orders, df_wishlist)
print(f"Total interactions: {len(df_interactions)}")
print(df_interactions.head(10))

In [ ]:
import implicit
from scipy.sparse import csr_matrix

# Map user_id và product_id sang index liên tục
user_ids = df_interactions['user_id'].unique()
product_ids = df_interactions['product_id'].unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
product2idx = {p: i for i, p in enumerate(product_ids)}
idx2product = {i: p for p, i in product2idx.items()}
idx2user = {i: u for u, i in user2idx.items()}

rows = df_interactions['user_id'].map(user2idx)
cols = df_interactions['product_id'].map(product2idx)
data = df_interactions['score'].values

# Item-user matrix (implicit ALS cần item x user)
item_user_matrix = csr_matrix(
    (data, (cols, rows)),
    shape=(len(product_ids), len(user_ids))
)

# Train ALS model - xịn hơn SVD, tốt cho implicit feedback
model_als = implicit.als.AlternatingLeastSquares(
    factors=64,           # 64 latent factors
    regularization=0.1,
    iterations=30,
    calculate_training_loss=True
)
model_als.fit(item_user_matrix)
print("✅ ALS trained!")
print(f"User factors shape: {model_als.user_factors.shape}")
print(f"Item factors shape: {model_als.item_factors.shape}")

In [ ]:
# Tạo text đầy đủ cho mỗi sản phẩm để embed
def build_product_text(row):
    parts = []
    if pd.notna(row.get('name')):
        parts.append(f"Tên: {row['name']}")
    if pd.notna(row.get('description')):
        parts.append(f"Mô tả: {row['description']}")
    if pd.notna(row.get('ingredients')):
        parts.append(f"Nguyên liệu: {row['ingredients']}")
    if pd.notna(row.get('nutrition_info')):
        parts.append(f"Dinh dưỡng: {row['nutrition_info']}")
    if pd.notna(row.get('tags')):
        parts.append(f"Tags: {row['tags']}")
    return " | ".join(parts)

df_products['full_text'] = df_products.apply(build_product_text, axis=1)
print("Sample text:")
print(df_products['full_text'].iloc[0])

# Dùng multilingual model - hiểu tiếng Việt tốt
print("\n⏳ Loading embedding model...")
embed_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("⏳ Generating embeddings...")
product_texts = df_products['full_text'].tolist()
product_embeddings = embed_model.encode(
    product_texts,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True  # Normalize để cosine similarity = dot product
)
print(f"✅ Embeddings shape: {product_embeddings.shape}")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Map product_id → embedding index
product_id_to_embed_idx = {
    row['id']: idx for idx, row in df_products.iterrows()
}

def build_user_profile(user_id, df_interactions, df_orders, df_wishlist):
    """
    Tạo user profile vector = weighted average của các sản phẩm đã tương tác
    Weight = interaction_score (mua nhiều, đánh giá cao = weight cao)
    """
    user_inters = df_interactions[df_interactions['user_id'] == user_id]

    if user_inters.empty:
        return None

    weighted_vecs = []
    weights = []

    for _, row in user_inters.iterrows():
        pid = row['product_id']
        if pid in product_id_to_embed_idx:
            embed_idx = product_id_to_embed_idx[pid]
            weighted_vecs.append(product_embeddings[embed_idx] * row['score'])
            weights.append(row['score'])

    if not weighted_vecs:
        return None

    profile = np.sum(weighted_vecs, axis=0) / sum(weights)
    # Normalize
    norm = np.linalg.norm(profile)
    if norm > 0:
        profile = profile / norm
    return profile

# Tạo profile cho tất cả users
user_profiles = {}
for uid in user_ids:
    profile = build_user_profile(uid, df_interactions, df_orders, df_wishlist)
    if profile is not None:
        user_profiles[uid] = profile

print(f"✅ Built profiles for {len(user_profiles)} users")

In [ ]:
def get_hybrid_recommendations(
    user_id,
    top_k=20,
    cf_weight=0.45,
    cb_weight=0.35,
    popularity_weight=0.20,
    diversity_lambda=0.3,
    final_k=10
):
    seen_products = set(
        df_interactions[df_interactions['user_id'] == user_id]['product_id'].tolist()
    )

    all_scores = {}

    # ============ 1. Collaborative Filtering ============
    if user_id in user2idx:
        user_idx = user2idx[user_id]
        user_items = item_user_matrix.T.tocsr()[user_idx]

        # FIX: bỏ filter_already_liked, dùng filter_items thay thế
        recommended_items, cf_scores = model_als.recommend(
            user_idx,
            user_items,
            N=len(product_ids),
            filter_items=[product2idx[p] for p in seen_products if p in product2idx]
        )

        if len(cf_scores) > 0:
            cf_min, cf_max = cf_scores.min(), cf_scores.max()
            if cf_max > cf_min:
                cf_scores_norm = (cf_scores - cf_min) / (cf_max - cf_min)
            else:
                cf_scores_norm = np.ones_like(cf_scores) * 0.5

            for item_idx, score in zip(recommended_items, cf_scores_norm):
                pid = idx2product[item_idx]
                if pid not in seen_products:
                    all_scores.setdefault(pid, {})['cf'] = float(score)

    # ============ 2. Content-Based ============
    if user_id in user_profiles:
        user_vec = user_profiles[user_id].reshape(1, -1)
        cb_scores = cosine_similarity(user_vec, product_embeddings)[0]

        for idx, row in df_products.iterrows():
            pid = int(row['id'])
            if pid not in seen_products:
                all_scores.setdefault(pid, {})['cb'] = float(cb_scores[idx])

    # ============ 3. Popularity ============
    pop_vals = df_products.apply(
        lambda r: float(r['rating']) * np.log1p(float(r['total_reviews'])), axis=1
    )
    max_pop = pop_vals.max() if pop_vals.max() > 0 else 1.0

    for idx, row in df_products.iterrows():
        pid = int(row['id'])
        if pid not in seen_products:
            pop = pop_vals[idx] / max_pop
            all_scores.setdefault(pid, {})['pop'] = float(pop)

    # ============ 4. Tổng hợp hybrid score ============
    final_scores = []
    for pid, scores in all_scores.items():
        cf_s  = scores.get('cf', 0.0)
        cb_s  = scores.get('cb', 0.0)
        pop_s = scores.get('pop', 0.0)

        # Cold start: user chưa có trong ALS
        if user_id not in user2idx:
            hybrid = cb_s * 0.6 + pop_s * 0.4
        else:
            hybrid = cf_weight * cf_s + cb_weight * cb_s + popularity_weight * pop_s

        final_scores.append((pid, hybrid))

    final_scores.sort(key=lambda x: x[1], reverse=True)
    top_candidates = final_scores[:top_k]

    # ============ 5. MMR Diversity Re-ranking ============
    selected = []
    candidate_ids = [x[0] for x in top_candidates]
    score_dict   = {x[0]: x[1] for x in top_candidates}

    pid_to_emb = {}
    for _, row in df_products.iterrows():
        pid_to_emb[int(row['id'])] = product_embeddings[_]

    pid_to_shop = {}
    for _, row in df_products.iterrows():
        pid_to_shop[int(row['id'])] = int(row['shop_id'])

    while len(selected) < final_k and candidate_ids:
        if not selected:
            best = candidate_ids[0]
        else:
            best      = None
            best_mmr  = -999
            sel_embs  = np.array([pid_to_emb[p] for p in selected if p in pid_to_emb])
            sel_shops = [pid_to_shop.get(p, -1) for p in selected]

            for pid in candidate_ids:
                relevance = score_dict[pid]

                # Similarity với các món đã chọn
                if len(sel_embs) > 0 and pid in pid_to_emb:
                    sims    = cosine_similarity(pid_to_emb[pid].reshape(1,-1), sel_embs)[0]
                    max_sim = float(sims.max())
                else:
                    max_sim = 0.0

                # Penalty cùng shop
                same_shop_count = sel_shops.count(pid_to_shop.get(pid, -1))
                shop_penalty    = same_shop_count * 0.15

                mmr = (1 - diversity_lambda) * relevance - diversity_lambda * max_sim - shop_penalty

                if mmr > best_mmr:
                    best_mmr = mmr
                    best     = pid

        if best is not None:
            selected.append(best)
            candidate_ids.remove(best)

    return selected

# Test
test_user = int(user_ids[0])
recs = get_hybrid_recommendations(test_user, final_k=10)
print(f"✅ Recommendations for user {test_user}:")
for pid in recs:
    prod = df_products[df_products['id'] == pid]
    if len(prod) > 0:
        r = prod.iloc[0]
        print(f"  [{pid}] {r['name']} | shop={int(r['shop_id'])} | rating={r['rating']}")

In [ ]:
import os
import pickle
import json
import numpy as np

# =========================
# 1. PRECOMPUTE RECOMMENDATIONS
# =========================
print("⏳ Precomputing recommendations for all users...")

all_recommendations = {}
for uid in user_ids:
    recs = get_hybrid_recommendations(uid, final_k=10)
    all_recommendations[int(uid)] = [int(p) for p in recs]

print(f"✅ Computed for {len(all_recommendations)} users")

# =========================
# 2. SAVE FILES
# =========================
os.makedirs('/content/model_output', exist_ok=True)

# ---- ALS model ----
with open('/content/model_output/als_model.pkl', 'wb') as f:
    pickle.dump(model_als, f)

# ---- Embeddings ----
np.save('/content/model_output/product_embeddings.npy', product_embeddings)

# ---- Mappings (fix numpy types) ----
with open('/content/model_output/mappings.pkl', 'wb') as f:
    pickle.dump({
        'user2idx': {int(k): int(v) for k, v in user2idx.items()},
        'product2idx': {int(k): int(v) for k, v in product2idx.items()},
        'idx2product': {int(k): int(v) for k, v in idx2product.items()},
        'idx2user': {int(k): int(v) for k, v in idx2user.items()},
        'product_id_to_embed_idx': {
            int(k): int(v) for k, v in product_id_to_embed_idx.items()
        },
        'user_profiles': {
            int(k): v.tolist() if isinstance(v, np.ndarray) else v
            for k, v in user_profiles.items()
        },
    }, f)

# ---- Precomputed recommendations (JSON-safe) ----
recs_serializable = {
    str(int(uid)): [int(pid) for pid in pids]
    for uid, pids in all_recommendations.items()
}

with open('/content/model_output/precomputed_recs.json', 'w') as f:
    json.dump(recs_serializable, f, ensure_ascii=False)

# ---- Products CSV ----
df_products.to_csv('/content/model_output/products_indexed.csv', index=False)

# =========================
# 3. VERIFY FILES
# =========================
print("\n✅ Files saved:")
for fname in os.listdir('/content/model_output'):
    size = os.path.getsize(f'/content/model_output/{fname}')
    print(f"  {fname}: {size/1024:.1f} KB")

# Preview JSON
with open('/content/model_output/precomputed_recs.json') as f:
    preview = json.load(f)

print(f"\n✅ precomputed_recs.json: {len(preview)} users")
for uid, pids in list(preview.items())[:2]:
    print(f"  user {uid} → {pids}")

# =========================
# 4. ZIP & DOWNLOAD
# =========================
!zip -r /content/model_output.zip /content/model_output/
print("\n✅ Zipped!")

from google.colab import files
files.download('/content/model_output.zip')